# Laya medical smoke — middle-layer FULL FT (no LoRA)

Freeze early/late ModernBERT layers; **full fine-tune middle third** + decision head. Same data/eval as prior smokes.

**Settings:** GPU T4 x2, Internet On.

In [ ]:
import os, sys, platform, subprocess, shutil
from pathlib import Path
import torch
print('python', sys.version)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
n = torch.cuda.device_count()
print('n_gpu', n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}', p.name, f'{p.total_memory/1e9:.1f}GB')
if n < 1:
    raise SystemExit('No GPU')
os.environ['USE_TF'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
%pip -q install -U 'laya>=0.3.7' 'transformers>=4.48.0' 'datasets>=3.0.0' safetensors huggingface_hub accelerate scipy pyarrow pandas tabulate PyYAML

In [ ]:
REPO = 'https://github.com/Priyanshu-5257/laya-medical-finetune.git'
REPO_DIR = Path('/kaggle/working/repo')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(['git', 'clone', '--branch', 'main', '--depth', '1', REPO, str(REPO_DIR)])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'])

In [ ]:
def run(cmd, cwd=None, env=None):
    print('+', *cmd, flush=True)
    e = os.environ.copy()
    if env: e.update(env)
    p = subprocess.Popen(cmd, cwd=cwd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(rc)

run(['bash', 'scripts/run_smoke_middle_ft.sh'], cwd=str(REPO_DIR), env={
    'WORK': '/kaggle/working',
    'CONFIG': 'configs/smoke_middle_ft.yaml',
})

In [ ]:
for name in ['summary_middle_ft.json', 'eval_report_middle_ft.json']:
    path = Path('/kaggle/working') / name
    print('====', name, '====')
    print(path.read_text() if path.exists() else 'MISSING')